# Iodine Clock Reaction — Data Collection

Record **HSV color vs. time** curves during iodine clock reactions using a USB camera.

**Workflow:**
1. Enter an experiment name (used as the filename for saved files)
2. Adjust **Camera ID** and **ROI** (Region of Interest) to frame the reaction vessel
3. Click **Start Recording** — the camera opens and data collection begins
4. Click **Stop Recording** — recording stops and the camera closes
5. Click **Save** — writes a `.avi` video and a `.xlsx` Excel file containing HSV data

> **HSV scale used (standard, NOT OpenCV):** $$H \in [0, 360°]$$, $$S \in [0, 100\%]$$, $$V \in [0, 100\%]$$

In [1]:
import cv2
import numpy as np
import threading
import time
import os

import ipywidgets as widgets
from IPython.display import display
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

In [2]:
class IodineClockRecorder:
    """Records iodine clock reaction color changes via a USB camera."""

    def __init__(self, color_cam_id=1, target_fps=15):
        self.color_cam_id = color_cam_id
        self.target_fps = target_fps

        self._cap = None
        self._running = False
        self._recording = False
        self._thread = None

        # ROI coordinates (pixels)
        self.roi_x1 = 200
        self.roi_y1 = 200
        self.roi_x2 = 400
        self.roi_y2 = 400

        # Auto-stop condition: HSV range + matched pixel ratio
        self.auto_stop_enabled = True
        self.hsv_lower = [45.0, 20.0, 20.0]   # standard scale: H(0-360), S/V(0-100)
        self.hsv_upper = [80.0, 100.0, 100.0]
        self.trigger_ratio = 0.6
        self._triggered = False
        self.trigger_time_s = None

        # Delay measurement and compensation
        self.compensate_startup_delay = True
        self.startup_delay_s = 0.0
        self._click_time = None

        # Recorded data
        self.hsv_data = []
        self._start_time = None
        self._video_writer = None
        self._video_path = None

        # UI widget references (assigned externally)
        self.image_widget = None
        self.status_label = None

    # ------------------------------------------------------------------ helpers
    def _update_status(self, msg):
        if self.status_label is not None:
            self.status_label.value = f"<b>Status:</b> {msg}"

    def _std_hsv_to_cv(self, h, s, v):
        """Convert standard HSV to OpenCV HSV scale."""
        h_cv = int(np.clip(round(h / 2.0), 0, 179))
        s_cv = int(np.clip(round(s * 2.55), 0, 255))
        v_cv = int(np.clip(round(v * 2.55), 0, 255))
        return np.array([h_cv, s_cv, v_cv], dtype=np.uint8)

    def _safe_roi(self, frame):
        fh, fw = frame.shape[:2]
        x1 = int(np.clip(min(self.roi_x1, self.roi_x2), 0, fw - 1))
        x2 = int(np.clip(max(self.roi_x1, self.roi_x2), x1 + 1, fw))
        y1 = int(np.clip(min(self.roi_y1, self.roi_y2), 0, fh - 1))
        y2 = int(np.clip(max(self.roi_y1, self.roi_y2), y1 + 1, fh))
        return x1, y1, x2, y2

    def _avg_hsv_in_roi(self, frame):
        """Return average (H, S, V) inside the ROI in standard scale."""
        x1, y1, x2, y2 = self._safe_roi(frame)
        roi = frame[y1:y2, x1:x2]
        hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

        h_cv = float(np.mean(hsv[:, :, 0]))
        s_cv = float(np.mean(hsv[:, :, 1]))
        v_cv = float(np.mean(hsv[:, :, 2]))

        # OpenCV HSV (H:0-179, S:0-255, V:0-255) -> standard (H:0-360, S:0-100, V:0-100)
        return round(h_cv * 2.0, 2), round(s_cv / 2.55, 2), round(v_cv / 2.55, 2)

    def _ratio_in_hsv_range(self, frame):
        """Return pixel ratio in ROI that falls within [hsv_lower, hsv_upper]."""
        x1, y1, x2, y2 = self._safe_roi(frame)
        roi = frame[y1:y2, x1:x2]
        hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

        lower_cv = self._std_hsv_to_cv(*self.hsv_lower)
        upper_cv = self._std_hsv_to_cv(*self.hsv_upper)
        mask = cv2.inRange(hsv, lower_cv, upper_cv)
        matched = cv2.countNonZero(mask)
        total = mask.size if mask.size > 0 else 1
        return matched / float(total)

    # ----------------------------------------------------------- camera thread
    def _loop(self):
        interval = 1.0 / self.target_fps

        while self._running:
            t0 = time.time()
            ret, frame = self._cap.read()
            if not ret or frame is None:
                self._update_status("Camera read error!")
                break

            h, s, v = self._avg_hsv_in_roi(frame)
            match_ratio = self._ratio_in_hsv_range(frame)

            # Display frame (with overlay)
            disp = frame.copy()
            x1, y1, x2, y2 = self._safe_roi(disp)
            cv2.rectangle(disp, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(disp, f"H:{h:.1f}  S:{s:.1f}  V:{v:.1f}",
                        (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
            cv2.putText(disp,
                        f"Ratio:{match_ratio:.2f}  Thr:{self.trigger_ratio:.2f}",
                        (10, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)
            cv2.putText(disp,
                        f"Delay:{self.startup_delay_s:.3f}s",
                        (10, 84), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (180, 220, 255), 2)

            if self._recording:
                elapsed_raw = time.time() - self._start_time
                elapsed = elapsed_raw + self.startup_delay_s if self.compensate_startup_delay else elapsed_raw

                self.hsv_data.append(
                    {
                        "time_s": round(elapsed, 3),
                        "time_raw_s": round(elapsed_raw, 3),
                        "startup_delay_s": round(self.startup_delay_s, 3),
                        "H": h,
                        "S": s,
                        "V": v,
                        "ratio": round(match_ratio, 4),
                    }
                )
                if self._video_writer is not None:
                    self._video_writer.write(frame)   # raw frame (no overlay)

                # REC indicator on the display copy only
                cv2.circle(disp, (disp.shape[1] - 30, 30), 10, (0, 0, 255), -1)
                cv2.putText(disp, f"REC  {elapsed:.1f}s",
                            (10, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

                if self.auto_stop_enabled and match_ratio >= self.trigger_ratio and not self._triggered:
                    self._triggered = True
                    self.trigger_time_s = round(elapsed, 3)
                    self._recording = False
                    self._running = False
                    self._update_status(
                        f"Auto-stopped at {self.trigger_time_s:.3f}s (ratio={match_ratio:.2f}, delay={self.startup_delay_s:.3f}s). Click Save."
                    )

            ok, jpg = cv2.imencode(".jpg", disp, [cv2.IMWRITE_JPEG_QUALITY, 85])
            if ok and self.image_widget is not None:
                self.image_widget.value = jpg.tobytes()

            dt = time.time() - t0
            if dt < interval:
                time.sleep(interval - dt)

        # cleanup on thread exit
        if self._video_writer is not None:
            self._video_writer.release()
            self._video_writer = None
        if self._cap is not None:
            self._cap.release()
            self._cap = None

    # -------------------------------------------------------------- public API
    def start_recording(self, exp_name, save_dir="."):
        self._click_time = time.time()
        if self._recording:
            self._update_status("Already recording!")
            return
        if not exp_name or not exp_name.strip():
            self._update_status("Please enter an experiment name first!")
            return

        self.hsv_data = []
        self._triggered = False
        self.trigger_time_s = None
        self.startup_delay_s = 0.0

        self._cap = cv2.VideoCapture(self.color_cam_id)
        if not self._cap.isOpened():
            self._update_status(f"Cannot open camera (ID={self.color_cam_id})!")
            return

        ret, test_frame = self._cap.read()
        if not ret or test_frame is None:
            self._update_status("Cannot read from camera!")
            self._cap.release()
            self._cap = None
            return

        fh, fw = test_frame.shape[:2]
        os.makedirs(save_dir, exist_ok=True)
        self._video_path = os.path.join(save_dir, f"{exp_name.strip()}.avi")
        fourcc = cv2.VideoWriter_fourcc(*"XVID")
        self._video_writer = cv2.VideoWriter(
            self._video_path, fourcc, self.target_fps, (fw, fh)
        )

        self._recording = True
        self._running = True
        self._start_time = time.time()
        self.startup_delay_s = max(0.0, self._start_time - self._click_time)

        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()
        self._update_status(
            f"Recording '{exp_name.strip()}' ... startup delay={self.startup_delay_s:.3f}s"
        )

    def stop_recording(self):
        if not self._recording and not self._running:
            self._update_status("Not recording.")
            return

        self._recording = False
        self._running = False
        if self._thread is not None:
            self._thread.join(timeout=5)
            self._thread = None

        n = len(self.hsv_data)
        dur = self.hsv_data[-1]["time_s"] if self.hsv_data else 0
        self._update_status(f"Stopped. {n} data points over {dur:.1f} s.  "
                            f"Delay={self.startup_delay_s:.3f}s  "
                            f"Video saved to {self._video_path}")

    def save_data(self, exp_name, save_dir="."):
        if self._recording:
            self._update_status("Stop recording before saving!")
            return False
        if not exp_name or not exp_name.strip():
            self._update_status("Please enter an experiment name!")
            return False
        if not self.hsv_data:
            self._update_status("No recorded data to save!")
            return False

        os.makedirs(save_dir, exist_ok=True)
        xlsx_path = os.path.join(save_dir, f"{exp_name.strip()}.xlsx")

        df = pd.DataFrame(self.hsv_data)
        df = df[["time_s", "time_raw_s", "startup_delay_s", "H", "S", "V", "ratio"]]
        df.columns = [
            "Time Compensated (s)",
            "Time Raw (s)",
            "Startup Delay (s)",
            "H (0-360)",
            "S (0-100)",
            "V (0-100)",
            "Matched Ratio",
        ]
        df.to_excel(xlsx_path, index=False, sheet_name="HSV Data")

        self._update_status(
            f"Saved!  Video -> {self._video_path}  |  Excel -> {xlsx_path}  |  Delay={self.startup_delay_s:.3f}s"
        )
        return True

In [3]:
recorder = IodineClockRecorder(color_cam_id=1, target_fps=15)

# -- widgets ------------------------------------------------------------------
w_lay = widgets.Layout(width="300px")
short_lay = widgets.Layout(width="180px")
desc_style = {"description_width": "90px"}
roi_style = {"description_width": "60px"}
hsv_style = {"description_width": "70px"}

exp_name_w = widgets.Text(
    value="", placeholder="e.g. iodine_0.1M_trial1",
    description="Exp Name:", layout=w_lay, style=desc_style,
 )
cam_id_w = widgets.IntText(
    value=1, description="Cam ID:", layout=short_lay, style=desc_style,
 )

roi_x1_w = widgets.IntText(value=200, description="ROI x1:", layout=short_lay, style=roi_style)
roi_y1_w = widgets.IntText(value=200, description="ROI y1:", layout=short_lay, style=roi_style)
roi_x2_w = widgets.IntText(value=400, description="ROI x2:", layout=short_lay, style=roi_style)
roi_y2_w = widgets.IntText(value=400, description="ROI y2:", layout=short_lay, style=roi_style)

auto_stop_w = widgets.Checkbox(value=True, description="Auto Stop", indent=False)
delay_comp_w = widgets.Checkbox(value=True, description="Delay Compensation", indent=False)
ratio_w = widgets.FloatSlider(
    value=0.60, min=0.0, max=1.0, step=0.01, description="Ratio:",
    continuous_update=False, readout_format=".2f", layout=widgets.Layout(width="320px")
)

h_low_w = widgets.BoundedFloatText(value=45.0, min=0.0, max=360.0, step=1.0, description="H low", layout=short_lay, style=hsv_style)
s_low_w = widgets.BoundedFloatText(value=20.0, min=0.0, max=100.0, step=1.0, description="S low", layout=short_lay, style=hsv_style)
v_low_w = widgets.BoundedFloatText(value=20.0, min=0.0, max=100.0, step=1.0, description="V low", layout=short_lay, style=hsv_style)
h_up_w = widgets.BoundedFloatText(value=80.0, min=0.0, max=360.0, step=1.0, description="H up", layout=short_lay, style=hsv_style)
s_up_w = widgets.BoundedFloatText(value=100.0, min=0.0, max=100.0, step=1.0, description="S up", layout=short_lay, style=hsv_style)
v_up_w = widgets.BoundedFloatText(value=100.0, min=0.0, max=100.0, step=1.0, description="V up", layout=short_lay, style=hsv_style)

btn_lay = widgets.Layout(width="180px", height="36px")
start_btn = widgets.Button(description="Start Recording", button_style="success", layout=btn_lay)
stop_btn  = widgets.Button(description="Stop Recording",  button_style="danger",  layout=btn_lay)
save_btn  = widgets.Button(description="Save",            button_style="info",    layout=btn_lay)

recorder.image_widget  = widgets.Image(format="jpeg", width=640, height=480)
recorder.status_label  = widgets.HTML("<b>Status:</b> Ready")
delay_label = widgets.HTML("<b>Measured Delay:</b> 0.000 s")
output_area = widgets.Output()

# -- callbacks ----------------------------------------------------------------
def _sync_params():
    recorder.color_cam_id = cam_id_w.value
    recorder.roi_x1 = roi_x1_w.value
    recorder.roi_y1 = roi_y1_w.value
    recorder.roi_x2 = roi_x2_w.value
    recorder.roi_y2 = roi_y2_w.value

    recorder.auto_stop_enabled = auto_stop_w.value
    recorder.compensate_startup_delay = delay_comp_w.value
    recorder.trigger_ratio = float(ratio_w.value)
    recorder.hsv_lower = [float(h_low_w.value), float(s_low_w.value), float(v_low_w.value)]
    recorder.hsv_upper = [float(h_up_w.value), float(s_up_w.value), float(v_up_w.value)]

def _refresh_delay_label():
    delay_label.value = f"<b>Measured Delay:</b> {recorder.startup_delay_s:.3f} s"

def on_start(b):
    _sync_params()
    recorder.start_recording(exp_name_w.value)
    _refresh_delay_label()

def on_stop(b):
    recorder.stop_recording()
    _refresh_delay_label()

def on_save(b):
    ok = recorder.save_data(exp_name_w.value)
    _refresh_delay_label()
    if ok:
        with output_area:
            output_area.clear_output(wait=True)
            df = pd.DataFrame(recorder.hsv_data)
            fig, axes = plt.subplots(4, 1, figsize=(10, 7), sharex=True)
            keys   = ["H", "S", "V", "ratio"]
            labels = ["H (0-360)", "S (0-100)", "V (0-100)", "Matched ratio"]
            colors = ["tab:red", "tab:green", "tab:blue", "tab:purple"]
            for ax, key, label, c in zip(axes, keys, labels, colors):
                ax.plot(df["time_s"], df[key], color=c, linewidth=1.2)
                ax.set_ylabel(label)
                ax.grid(True, alpha=0.3)
            axes[3].axhline(recorder.trigger_ratio, color="black", linestyle="--", linewidth=1)
            axes[-1].set_xlabel("Time (s)")
            fig.suptitle(f"HSV/Ratio vs Time - {exp_name_w.value}", fontsize=14)
            plt.tight_layout()
            plt.show()

start_btn.on_click(on_start)
stop_btn.on_click(on_stop)
save_btn.on_click(on_save)

# -- layout -------------------------------------------------------------------
settings_row = widgets.HBox([
    widgets.VBox([exp_name_w, cam_id_w]),
    widgets.VBox([
        widgets.HTML("<b>ROI (Region of Interest)</b>"),
        widgets.HBox([roi_x1_w, roi_y1_w]),
        widgets.HBox([roi_x2_w, roi_y2_w]),
    ]),
])

threshold_box = widgets.VBox([
    widgets.HTML("<b>Auto Stop Condition (HSV range + ratio)</b>"),
    widgets.HBox([auto_stop_w, delay_comp_w, ratio_w]),
    widgets.HBox([h_low_w, s_low_w, v_low_w]),
    widgets.HBox([h_up_w, s_up_w, v_up_w]),
])

buttons_row = widgets.HBox([start_btn, stop_btn, save_btn])

save_note = widgets.HTML(
    "<i style='color:#666; font-size:12px;'>"
    "Timer compensation uses measured startup delay from click to recorder ready.</i>"
)

ui = widgets.VBox([
    settings_row,
    widgets.HTML("<hr style='border:1px solid #ccc'>"),
    threshold_box,
    delay_label,
    widgets.HTML("<hr style='border:1px solid #ccc'>"),
    buttons_row,
    save_note,
    recorder.status_label,
    recorder.image_widget,
    output_area,
])

display(ui)